# Task 3: News Sentiment and Stock Price Correlation Analysis

This notebook quantifies the relationship between financial news sentiment and daily stock price returns using statistical methods.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.stats import pearsonr, spearmanr
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Download NLTK VADER lexicon (only needed once)
try:
    import nltk
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    import nltk
    nltk.download('vader_lexicon')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully")

## 1. Data Loading and Date Alignment

In [ ]:
# Load datasets
news_df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')
stock_df = pd.read_csv('../data/raw/AAPL.csv')

# Convert date columns
news_df['date'] = pd.to_datetime(news_df['date'])
stock_df['Date'] = pd.to_datetime(stock_df['Date'])

# Extract date only (without time) for news
news_df['date_only'] = news_df['date'].dt.date
stock_df['date_only'] = stock_df['Date'].dt.date

print(f"News dataset shape: {news_df.shape}")
print(f"Stock dataset shape: {stock_df.shape}")
print(f"News date range: {news_df['date'].min()} to {news_df['date'].max()}")
print(f"Stock date range: {stock_df['Date'].min()} to {stock_df['Date'].max()}")

In [ ]:
# Filter news for AAPL stock only (or handle multiple stocks)
# First, let's see what stocks are available
print("Unique stocks in news dataset:")
print(news_df['stock'].value_counts().head(10))

# For this analysis, we'll focus on AAPL news
aapl_news = news_df[news_df['stock'] == 'AAPL'].copy()
print(f"\nAAPL news articles: {len(aapl_news)}")

In [ ]:
# Calculate daily stock returns
stock_df['Daily_Return'] = stock_df['Close'].pct_change() * 100
stock_df['date_only'] = stock_df['Date'].dt.date

# Create a clean stock dataframe with date as index
stock_clean = stock_df[['date_only', 'Close', 'Daily_Return']].copy()
stock_clean = stock_clean.dropna()

print("Stock data with returns:")
print(stock_clean.head())

In [ ]:
# Handle weekend/holiday alignment
def align_to_trading_day(date, trading_days):
    """Align news date to the next trading day if published on weekend/holiday"""
    if date in trading_days:
        return date
    else:
        # Find the next trading day
        current_date = date
        for i in range(1, 8):  # Look up to 7 days ahead
            next_date = current_date + timedelta(days=i)
            if next_date in trading_days:
                return next_date
        return date  # Fallback

# Get set of trading days
trading_days = set(stock_clean['date_only'])

# Align news dates to trading days
aapl_news['trading_date'] = aapl_news['date_only'].apply(lambda x: align_to_trading_day(x, trading_days))

print(f"News articles after date alignment: {len(aapl_news)}")
print("Sample of aligned news dates:")
print(aapl_news[['date_only', 'trading_date', 'headline']].head())

## 2. Sentiment Analysis on Headlines

In [ ]:
# Initialize sentiment analyzers
vader = SentimentIntensityAnalyzer()

# Function to get sentiment scores
def get_sentiment_scores(text):
    # TextBlob sentiment
    blob = TextBlob(str(text))
    textblob_polarity = blob.sentiment.polarity
    textblob_subjectivity = blob.sentiment.subjectivity
    
    # VADER sentiment
    vader_scores = vader.polarity_scores(str(text))
    vader_compound = vader_scores['compound']
    
    return {
        'textblob_polarity': textblob_polarity,
        'textblob_subjectivity': textblob_subjectivity,
        'vader_compound': vader_compound,
        'vader_positive': vader_scores['pos'],
        'vader_negative': vader_scores['neg'],
        'vader_neutral': vader_scores['neu']
    }

# Apply sentiment analysis to headlines
print("Analyzing sentiment...")
sentiment_scores = aapl_news['headline'].apply(get_sentiment_scores)
sentiment_df = pd.DataFrame(sentiment_scores.tolist())

# Combine with news data
aapl_news_sentiment = pd.concat([aapl_news.reset_index(drop=True), sentiment_df], axis=1)

print("Sentiment analysis completed")
print("Sample sentiment scores:")
print(aapl_news_sentiment[['headline', 'textblob_polarity', 'vader_compound']].head())

In [ ]:
# Sentiment score distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# TextBlob Polarity
axes[0,0].hist(aapl_news_sentiment['textblob_polarity'], bins=30, alpha=0.7, color='skyblue')
axes[0,0].set_title('TextBlob Polarity Distribution')
axes[0,0].set_xlabel('Polarity (-1 to 1)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(aapl_news_sentiment['textblob_polarity'].mean(), color='red', linestyle='--', 
                label=f'Mean: {aapl_news_sentiment["textblob_polarity"].mean():.3f}')
axes[0,0].legend()

# VADER Compound
axes[0,1].hist(aapl_news_sentiment['vader_compound'], bins=30, alpha=0.7, color='lightcoral')
axes[0,1].set_title('VADER Compound Distribution')
axes[0,1].set_xlabel('Compound Score (-1 to 1)')
axes[0,1].set_ylabel('Frequency')
axes[0,1].axvline(aapl_news_sentiment['vader_compound'].mean(), color='red', linestyle='--',
                label=f'Mean: {aapl_news_sentiment["vader_compound"].mean():.3f}')
axes[0,1].legend()

# TextBlob vs VADER scatter
axes[1,0].scatter(aapl_news_sentiment['textblob_polarity'], aapl_news_sentiment['vader_compound'], 
                alpha=0.6, color='green')
axes[1,0].set_xlabel('TextBlob Polarity')
axes[1,0].set_ylabel('VADER Compound')
axes[1,0].set_title('TextBlob vs VADER Sentiment Scores')
axes[1,0].grid(True, alpha=0.3)

# Correlation between sentiment methods
correlation = aapl_news_sentiment['textblob_polarity'].corr(aapl_news_sentiment['vader_compound'])
axes[1,1].text(0.5, 0.5, f'Correlation between\nTextBlob and VADER:\n{correlation:.3f}', 
                ha='center', va='center', fontsize=16, transform=axes[1,1].transAxes,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
axes[1,1].set_xlim(0, 1)
axes[1,1].set_ylim(0, 1)
axes[1,1].axis('off')

plt.tight_layout()
plt.show()

## 3. Daily Aggregation and Correlation Analysis

In [ ]:
# Aggregate sentiment scores by trading date
daily_sentiment = aapl_news_sentiment.groupby('trading_date').agg({
    'textblob_polarity': 'mean',
    'vader_compound': 'mean',
    'vader_positive': 'mean',
    'vader_negative': 'mean',
    'vader_neutral': 'mean',
    'headline': 'count'  # Number of articles per day
}).rename(columns={'headline': 'article_count'})

print(f"Daily sentiment data: {len(daily_sentiment)} days")
print("Sample daily sentiment:")
print(daily_sentiment.head())

In [ ]:
# Merge sentiment data with stock returns
merged_df = pd.merge(daily_sentiment, stock_clean, left_index=True, right_on='date_only', how='inner')
merged_df.set_index('date_only', inplace=True)

print(f"Merged dataset: {len(merged_df)} days")
print("\nMerged data sample:")
print(merged_df[['textblob_polarity', 'vader_compound', 'Daily_Return', 'article_count']].head())

In [ ]:
# Calculate correlation coefficients
correlations = {}

# Remove any rows with missing values
analysis_df = merged_df.dropna()

# Pearson correlations
correlations['textblob_pearson'] = pearsonr(analysis_df['textblob_polarity'], analysis_df['Daily_Return'])
correlations['vader_pearson'] = pearsonr(analysis_df['vader_compound'], analysis_df['Daily_Return'])

# Spearman correlations
correlations['textblob_spearman'] = spearmanr(analysis_df['textblob_polarity'], analysis_df['Daily_Return'])
correlations['vader_spearman'] = spearmanr(analysis_df['vader_compound'], analysis_df['Daily_Return'])

print("Correlation Analysis Results:")
print("="*50)
for key, (corr, p_value) in correlations.items():
    method, corr_type = key.split('_')
    print(f"{method.capitalize()} {corr_type.capitalize()} Correlation: {corr:.4f} (p-value: {p_value:.4f})")

## 4. Visualization of Correlation Results

In [ ]:
# Scatter plots with regression lines
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# TextBlob Polarity vs Returns
axes[0].scatter(analysis_df['textblob_polarity'], analysis_df['Daily_Return'], 
                alpha=0.6, color='blue', s=50)

# Add regression line
z = np.polyfit(analysis_df['textblob_polarity'], analysis_df['Daily_Return'], 1)
p = np.poly1d(z)
axes[0].plot(analysis_df['textblob_polarity'], p(analysis_df['textblob_polarity']), 
             "r--", alpha=0.8, linewidth=2)

axes[0].set_xlabel('TextBlob Polarity')
axes[0].set_ylabel('Daily Return (%)')
axes[0].set_title(f'TextBlob Sentiment vs Stock Returns\nCorrelation: {correlations["textblob_pearson"][0]:.3f}')
axes[0].grid(True, alpha=0.3)

# VADER Compound vs Returns
axes[1].scatter(analysis_df['vader_compound'], analysis_df['Daily_Return'], 
                alpha=0.6, color='red', s=50)

# Add regression line
z = np.polyfit(analysis_df['vader_compound'], analysis_df['Daily_Return'], 1)
p = np.poly1d(z)
axes[1].plot(analysis_df['vader_compound'], p(analysis_df['vader_compound']), 
             "r--", alpha=0.8, linewidth=2)

axes[1].set_xlabel('VADER Compound Score')
axes[1].set_ylabel('Daily Return (%)')
axes[1].set_title(f'VADER Sentiment vs Stock Returns\nCorrelation: {correlations["vader_pearson"][0]:.3f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sentiment classification and average returns
def classify_sentiment(score, method='vader'):
    if method == 'vader':
        if score > 0.05:
            return 'Positive'
        elif score < -0.05:
            return 'Negative'
        else:
            return 'Neutral'
    else:  # textblob
        if score > 0.1:
            return 'Positive'
        elif score < -0.1:
            return 'Negative'
        else:
            return 'Neutral'

# Classify sentiment
analysis_df['textblob_sentiment'] = analysis_df['textblob_polarity'].apply(lambda x: classify_sentiment(x, 'textblob'))
analysis_df['vader_sentiment'] = analysis_df['vader_compound'].apply(lambda x: classify_sentiment(x, 'vader'))

# Calculate average returns by sentiment category
textblob_returns = analysis_df.groupby('textblob_sentiment')['Daily_Return'].agg(['mean', 'count', 'std'])
vader_returns = analysis_df.groupby('vader_sentiment')['Daily_Return'].agg(['mean', 'count', 'std'])

print("Average Daily Returns by Sentiment Category:")
print("\nTextBlob Classification:")
print(textblob_returns)
print("\nVADER Classification:")
print(vader_returns)

In [ ]:
# Bar charts of average returns by sentiment
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# TextBlob results
textblob_returns['mean'].plot(kind='bar', ax=axes[0], color=['red', 'gray', 'green'], alpha=0.7)
axes[0].set_title('Average Daily Returns by TextBlob Sentiment')
axes[0].set_ylabel('Average Daily Return (%)')
axes[0].set_xlabel('Sentiment Category')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(True, alpha=0.3)

# Add value labels on bars
for i, v in enumerate(textblob_returns['mean']):
    axes[0].text(i, v + 0.01, f'{v:.3f}%', ha='center', va='bottom')

# VADER results
vader_returns['mean'].plot(kind='bar', ax=axes[1], color=['red', 'gray', 'green'], alpha=0.7)
axes[1].set_title('Average Daily Returns by VADER Sentiment')
axes[1].set_ylabel('Average Daily Return (%)')
axes[1].set_xlabel('Sentiment Category')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(True, alpha=0.3)

# Add value labels on bars
for i, v in enumerate(vader_returns['mean']):
    axes[1].text(i, v + 0.01, f'{v:.3f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 5. Time Series Analysis

In [ ]:
# Time series plot of sentiment and returns
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Plot sentiment scores over time
axes[0].plot(analysis_df.index, analysis_df['vader_compound'], label='VADER Compound', color='blue', alpha=0.7)
axes[0].plot(analysis_df.index, analysis_df['textblob_polarity'], label='TextBlob Polarity', color='red', alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0].set_title('News Sentiment Over Time')
axes[0].set_ylabel('Sentiment Score')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot daily returns
axes[1].plot(analysis_df.index, analysis_df['Daily_Return'], color='green', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Daily Stock Returns')
axes[1].set_ylabel('Daily Return (%)')
axes[1].grid(True, alpha=0.3)

# Plot article count
axes[2].bar(analysis_df.index, analysis_df['article_count'], alpha=0.6, color='orange')
axes[2].set_title('Number of News Articles per Day')
axes[2].set_ylabel('Article Count')
axes[2].set_xlabel('Date')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Statistical Analysis and Interpretation

In [ ]:
# Additional statistical tests
from scipy import stats

# Test if positive sentiment days have significantly different returns than negative sentiment days
positive_returns = analysis_df[analysis_df['vader_sentiment'] == 'Positive']['Daily_Return']
negative_returns = analysis_df[analysis_df['vader_sentiment'] == 'Negative']['Daily_Return']
neutral_returns = analysis_df[analysis_df['vader_sentiment'] == 'Neutral']['Daily_Return']

print("Statistical Tests:")
print("="*40)

# T-test between positive and negative sentiment days
if len(positive_returns) > 0 and len(negative_returns) > 0:
    t_stat, p_val = stats.ttest_ind(positive_returns, negative_returns)
    print(f"T-test (Positive vs Negative): t-stat={t_stat:.4f}, p-value={p_val:.4f}")

# T-test between positive and neutral sentiment days
if len(positive_returns) > 0 and len(neutral_returns) > 0:
    t_stat, p_val = stats.ttest_ind(positive_returns, neutral_returns)
    print(f"T-test (Positive vs Neutral): t-stat={t_stat:.4f}, p-value={p_val:.4f}")

# ANOVA test for all three groups
if len(positive_returns) > 0 and len(negative_returns) > 0 and len(neutral_returns) > 0:
    f_stat, p_val = stats.f_oneway(positive_returns, negative_returns, neutral_returns)
    print(f"ANOVA (All Groups): F-stat={f_stat:.4f}, p-value={p_val:.4f}")

In [ ]:
# Lag analysis - does sentiment predict next day returns?
analysis_df['Next_Day_Return'] = analysis_df['Daily_Return'].shift(-1)

# Calculate lagged correlations
lagged_correlations = {}
lagged_df = analysis_df.dropna()

lagged_correlations['textblob_lagged'] = pearsonr(lagged_df['textblob_polarity'], lagged_df['Next_Day_Return'])
lagged_correlations['vader_lagged'] = pearsonr(lagged_df['vader_compound'], lagged_df['Next_Day_Return'])

print("Lagged Correlation Analysis (Sentiment vs Next Day Returns):")
print("="*60)
for key, (corr, p_value) in lagged_correlations.items():
    method = key.split('_')[0]
    print(f"{method.capitalize()} Lagged Correlation: {corr:.4f} (p-value: {p_value:.4f})")

print("\nComparison with Same-Day Correlations:")
print(f"TextBlob Same-Day: {correlations['textblob_pearson'][0]:.4f}")
print(f"VADER Same-Day: {correlations['vader_pearson'][0]:.4f}")

## 7. Final Results and Investment Strategy Implications

In [ ]:
# Summary of findings
summary_results = {
    'Analysis Period': f"{analysis_df.index.min()} to {analysis_df.index.max()}",
    'Total Trading Days': len(analysis_df),
    'Total News Articles': analysis_df['article_count'].sum(),
    'Avg Articles per Day': f"{analysis_df['article_count'].mean():.1f}",
    'TextBlob Correlation': f"{correlations['textblob_pearson'][0]:.4f} (p={correlations['textblob_pearson'][1]:.4f})",
    'VADER Correlation': f"{correlations['vader_pearson'][0]:.4f} (p={correlations['vader_pearson'][1]:.4f})",
    'Best Sentiment Method': 'TextBlob' if abs(correlations['textblob_pearson'][0]) > abs(correlations['vader_pearson'][0]) else 'VADER',
    'Positive Sentiment Avg Return': f"{vader_returns.loc['Positive', 'mean']:.4f}%",
    'Negative Sentiment Avg Return': f"{vader_returns.loc['Negative', 'mean']:.4f}%",
    'Neutral Sentiment Avg Return': f"{vader_returns.loc['Neutral', 'mean']:.4f}%"
}

print("CORRELATION ANALYSIS SUMMARY")
print("="*50)
for key, value in summary_results.items():
    print(f"{key}: {value}")

### Key Findings and Investment Strategy Recommendations:

#### 1. **Sentiment-Return Relationship**
- The correlation between news sentiment and stock returns is [weak/moderate/strong] at [value]
- [TextBlob/VADER] shows slightly better correlation with stock movements
- The relationship is [statistically significant/not significant] (p-value: [value])

#### 2. **Sentiment Classification Performance**
- Days with positive sentiment show average returns of [value]%
- Days with negative sentiment show average returns of [value]%
- The difference between positive and negative sentiment days is [statistically significant/not significant]

#### 3. **Predictive Power**
- Lagged correlation (sentiment predicting next-day returns) is [weaker/stronger] than same-day correlation
- This suggests [limited/improved] predictive capability for short-term trading

#### 4. **Investment Strategy Implications**

**Conservative Approach:**
- Use sentiment as a confirmation tool rather than primary signal
- Combine sentiment analysis with technical indicators for better results
- Focus on extreme sentiment readings (very positive or very negative)

**Active Trading Strategy:**
- Consider long positions on days with strongly positive sentiment
- Reduce exposure or consider short positions on days with strongly negative sentiment
- Monitor sentiment trends over multiple days rather than single-day readings

#### 5. **Limitations and Considerations**
- Analysis limited to [time period] and [stock symbol]
- Sentiment analysis may not capture context-specific financial terminology
- Market efficiency may limit arbitrage opportunities
- External factors (earnings, macro events) may override sentiment effects

#### 6. **Future Improvements**
- Expand analysis to multiple stocks and sectors
- Incorporate more sophisticated NLP models (BERT, financial-specific models)
- Consider sentiment momentum and trend analysis
- Include volume-based analysis and market-wide sentiment indicators

This analysis provides a foundation for sentiment-driven investment strategies, though real-world implementation should consider additional market factors and risk management principles.